# B+ → π+ π+ π−: isobar self-closure without CP

This notebook is the simplest non-CP closure baseline:

- the generator and fit use the **same isobaric model**;
- the toy is generated with the exact **accept-reject** sampler;
- only one $B^+$ sample is generated and fitted;
- the $\rho(770)$ coefficient is the global amplitude reference, fixed to $1+0i$;
- all other isobar coefficients float from their generated values;
- there is no efficiency, background, veto, smearing, or QMI.

A successful closure here validates the single-sample isobaric chain before replacing the scalar isobar by QMI.


In [ ]:
import numpy as np
import pandas as pd
import jax.numpy as jnp
import matplotlib.pyplot as plt

from dataclasses import dataclass

from dalitzplotfitter import (
    DecayChannel,
    DecayModel,
    FitSession,
    GounarisSakurai,
    Parameter,
    RealImag,
    RelativisticBreitWigner,
    Resonance,
    enable_x64,
    generate_signal_toy,
)

enable_x64()

SEED = 20260905
N_EVENTS = 250_000
TOY_METHOD = "accept-reject"
NORMALIZATION_METHOD = "gauss-legendre"

channel = DecayChannel("B+", ("pi+", "pi+", "pi-"))
M_PI = float(channel.daughter_masses[0])

print("Toy size:", N_EVENTS)
print("Toy method:", TOY_METHOD)
print("Normalization:", NORMALIZATION_METHOD)


## Isobaric truth

The generated coefficients are the CP-averaged Cartesian coefficients used in the neighboring closure studies,

\[
c_j = x_j + i y_j.
\]

The $\rho(770)$ defines the amplitude convention.


In [ ]:
TRUTH_COEFFICIENTS = {
    "rho770":    dict(x= 1.000, y= 0.000),
    "omega782":  dict(x= 0.091, y=-0.007),
    "f2_1270":   dict(x= 0.291, y= 0.204),
    "rho1450":   dict(x=-0.223, y= 0.191),
    "rho3_1690": dict(x= 0.073, y=-0.045),
    "sigma":     dict(x=-0.485, y= 0.284),
}

def truth_coefficient(name):
    p = TRUTH_COEFFICIENTS[name]
    return RealImag(p["x"], p["y"])


@dataclass(frozen=True)
class PaperSigmaPole:
    # LHCb Eq. (16): A_sigma(m) = 1 / (s_sigma - m^2)
    def __call__(self, mass, context):
        m = jnp.asarray(mass)
        pole = jnp.asarray(context.pole_mass) - 1j * jnp.asarray(context.pole_width)
        s_sigma = pole**2
        return 1.0 / (s_sigma - m**2)


def isobar_components(coefficient_factory):
    return [
        Resonance(
            "rho770", (0, 2), coefficient_factory("rho770"),
            mass=0.7708, width=0.1534, spin=1,
            lineshape=GounarisSakurai(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "omega782", (0, 2), coefficient_factory("omega782"),
            mass=0.78265, width=0.00849, spin=1,
            lineshape=RelativisticBreitWigner(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "f2_1270", (0, 2), coefficient_factory("f2_1270"),
            mass=1.2755, width=0.1867, spin=2,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho1450", (0, 2), coefficient_factory("rho1450"),
            mass=1.465, width=0.400, spin=1,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "rho3_1690", (0, 2), coefficient_factory("rho3_1690"),
            mass=1.6888, width=0.161, spin=3,
            resonance_radius=4.0, parent_radius=4.0,
        ),
        Resonance(
            "sigma", (0, 2), coefficient_factory("sigma"),
            mass=0.563, width=0.350, spin=0,
            lineshape=PaperSigmaPole(),
            resonance_radius=4.0, parent_radius=4.0,
        ),
    ]


generator = DecayModel(
    channel,
    isobar_components(truth_coefficient),
    normalize_components=True,
    normalization_method=NORMALIZATION_METHOD,
)

print("Generator normalization points:", generator.normalization_sample.size)


## Generate the isobaric toy with accept-reject

The optimized accept-reject path evaluates proposal candidates directly in Dalitz invariants and uses monitored local envelopes. Four-momenta are not retained because the fit and diagnostics below use only the three invariants.


In [ ]:
toy = generate_signal_toy(
    generator,
    N_EVENTS,
    seed=SEED,
    method=TOY_METHOD,
    include_momenta=False,
)

print("Generated events:", toy.size)
print("Toy method:", TOY_METHOD)
print("weights:", np.unique(np.asarray(toy.weights)))
print("momenta retained:", toy.p1 is not None)


In [ ]:
# Toy Dalitz plot and invariant-mass projections before any fit.
fig, axes = plt.subplots(1, 3, figsize=(16, 4.6), constrained_layout=True)

axes[0].hist2d(
    np.asarray(toy.s13),
    np.asarray(toy.s23),
    bins=180,
)
axes[0].set_xlabel(r"$s_{13}$ [GeV$^2$]")
axes[0].set_ylabel(r"$s_{23}$ [GeV$^2$]")
axes[0].set_title("isobaric toy Dalitz plot")

for ax, values, label in [
    (axes[1], np.sqrt(np.asarray(toy.s13)), r"$m_{13}$"),
    (axes[2], np.sqrt(np.asarray(toy.s23)), r"$m_{23}$"),
]:
    ax.hist(values, bins=100, histtype="step")
    ax.set_xlabel(label + " [GeV]")
    ax.set_ylabel("events")

plt.show()


## Identical isobaric fit model

The fit uses the same lineshapes, masses, widths, radii and component-normalization convention as the generator. Only the complex coefficients are fitted. The $\rho(770)$ remains fixed to $1+0i$ as the global reference.


In [ ]:
def floating_coefficient(name):
    truth = TRUTH_COEFFICIENTS[name]
    reference = name == "rho770"
    return RealImag(
        Parameter.coefficient(
            f"{name}.x",
            truth["x"],
            owner=name,
            fixed=reference,
            step=0.01,
        ),
        Parameter.coefficient(
            f"{name}.y",
            truth["y"],
            owner=name,
            fixed=reference,
            step=0.01,
        ),
    )


fit_model = DecayModel(
    channel,
    isobar_components(floating_coefficient),
    normalize_components=True,
    normalization_method=NORMALIZATION_METHOD,
)

session = FitSession(fit_model, toy)

print("Number of fit parameters:", len(session.parameters))
print("Free parameters:", sum(not p.fixed for p in session.parameters))
for parameter in session.parameters:
    print(
        f"{parameter.name:20s} "
        f"start={float(parameter.value): .6f} "
        f"fixed={parameter.fixed}"
    )


## Self-closure fit


In [ ]:
start_values = {
    parameter.name: float(parameter.value)
    for parameter in session.parameters
    if not parameter.fixed
}

result = session.fit(
    start_values=start_values,
    simplex=False,
    strategy=1,
    hesse=True,
    tolerance=1e-4,
    verbose=2,
)

fit_values = session.print_result(result)

print()
print("valid:", bool(result.valid))
print("EDM:", float(result.fmin.edm))
print("nfcn:", int(result.nfcn))


## Coefficient closure


In [ ]:
rows = []
for name, truth in TRUTH_COEFFICIENTS.items():
    for field in ("x", "y"):
        pname = f"{name}.{field}"
        parameter = next((p for p in session.parameters if p.name == pname), None)
        if parameter is None:
            continue

        fitted = float(fit_values[pname])
        error = 0.0 if parameter.fixed else float(result.errors[pname])
        pull = np.nan if error <= 0.0 else (fitted - truth[field]) / error
        rows.append({
            "component": name,
            "parameter": field,
            "truth": truth[field],
            "fit": fitted,
            "error": error,
            "pull": pull,
            "fixed": parameter.fixed,
        })

coefficient_table = pd.DataFrame(rows)
display(coefficient_table)


## Fit fractions and projections


In [ ]:
session.print_fit_fractions(
    result,
    acceptance_weighted=False,
    include_interference=False,
    precision=4,
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8), constrained_layout=True)

session.plot_projection(
    result,
    "s13",
    bins=80,
    projection_size=300_000,
    projection_seed=SEED + 700,
    ax=axes[0],
)
session.plot_projection(
    result,
    "s23",
    bins=80,
    projection_size=300_000,
    projection_seed=SEED + 701,
    ax=axes[1],
)

plt.show()


## Closure criteria

This baseline should show:

1. a valid coefficient-only fit with small EDM;
2. non-reference Cartesian coefficients statistically compatible with their generated values;
3. fit fractions consistent with the generator model;
4. smooth $s_{13}$ and $s_{23}$ projections describing the accept-reject toy.

Once this closes, notebook 23 tests the harder replacement of the isobaric scalar wave by QMI.
